# Model Training with Leakage


In [12]:
from pathlib import Path
import numpy as np
import h5py

import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, average_precision_score


In [ ]:

records = {}

# =====================================================
# LOAD SEIZURE FILES
# =====================================================

seizure_files = sorted(
    Path("../Features/Seizure").glob("chb02_*_seizure.mat")
)

for file in seizure_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_seizure", "")

    records[record_name] = (X, y)

    print(
        "[SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

# =====================================================
# LOAD NON-SEIZURE FILES
# =====================================================

# non_files = sorted(
#     Path("../Features/Non_Seizure").glob("chb02_*_features.mat")
# )

# for file in non_files:

#     with h5py.File(file, "r") as f:
#         X = f["all_feats"][:]

#     if X.shape[0] == 1530:
#         X = X.T

#     y = np.zeros(X.shape[0], dtype=int)

#     record_name = file.stem.replace("_features", "")

#     records[record_name] = (X, y)

#     print(
#         "[NON-SEIZURE]",
#         record_name,
#         X.shape,
#         y.shape,
#         "positives =", int(np.sum(y))
#     )

# print("\nTotal records:", len(records))

[SEIZURE] chb02_16+ (1437, 1530) (1437,) positives = 33
[SEIZURE] chb02_16 (380, 1530) (380,) positives = 33
[SEIZURE] chb02_19 (1437, 1530) (1437,) positives = 4
[NON-SEIZURE] chb02_13 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_14 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_15 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_17 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_18 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_20 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_21 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_22 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_23 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_24 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_25 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_26 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_27 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_28 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_29 (1

In [27]:
# =====================================================
# SPLIT SEIZURE / NON-SEIZURE RECORDS
# =====================================================
seizure_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) > 0
}

non_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) == 0
}


print("Seizure:", len(seizure_records))
print("Non seizure:", len(non_records))

Seizure: 3
Non seizure: 20


In [28]:
# =====================================================
# SELECT NON-SEIZURE RECORDS FOR TRAINING
# =====================================================

r = 0
random.seed(42)


selected_non = random.sample(
    list(non_records.keys()),
    r
)


print("Selected non seizure:")
print(selected_non)

Selected non seizure:
[]


In [29]:
def create_train_test(test_record):

    X_train = []
    y_train = []
    train_groups = []


    # -------------------------
    # seizure records
    # -------------------------

    for name, (X,y) in seizure_records.items():

        if name == test_record:
            continue


        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # -------------------------
    # non seizure records
    # -------------------------

    for name in selected_non:

        X,y = non_records[name]

        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # combine

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    train_groups = np.array(train_groups)



    # test

    X_test, y_test = seizure_records[test_record]


    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups
    )

In [30]:
X_all = np.vstack([X for X, y in seizure_records.values()])
y_all = np.hstack([y for X, y in seizure_records.values()])

## 1. Test loading files

In [32]:
import numpy as np
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, confusion_matrix
)

# =====================================================
# CONFIG — the ONLY things you change between runs
# =====================================================
SEED = 42
DATA_POOL = seizure_records   # <-- swap to `records` for the all-files condition
TUNE = True                   # <-- True = grid search; False = use FIXED_PARAMS
FIXED_PARAMS = {"k": 100, "C": 10, "gamma": "scale"}

param_grid = {
    "k": [10, 30, 50, 100],
    "C": [0.1, 1, 10,100],
    "gamma": ["scale", 0.01, 0.1],
}

# =====================================================
# BUILD DATA  (identical for every run on the same pool)
# =====================================================
X_all = np.vstack([X for X, y in DATA_POOL.values()])
y_all = np.hstack([y for X, y in DATA_POOL.values()]).astype(int)
print("Total windows:", X_all.shape)
print("Class counts :", np.bincount(y_all))

# =====================================================
# SPLIT — fixed 70/20/10, fixed seed, identical every run
# =====================================================
# 1. carve off test FIRST — 10% of everything, set aside
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.10, random_state=SEED, stratify=y_all
)

# 2. split the remaining 90% into train and val
#    val = 0.20/0.90 ≈ 0.2222 of trainval  -> 20% of the whole
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=2/9, random_state=SEED, stratify=y_trainval
)
print("Train/Val/Test:", X_train.shape[0], X_val.shape[0], X_test.shape[0])
print("Test positives:", int(y_test.sum()))


def build(params):
    return Pipeline([
        ("select", SelectKBest(f_classif, k=params["k"])),
        ("scale",  StandardScaler()),
        ("svm",    SVC(kernel="rbf", C=params["C"], gamma=params["gamma"],
                       class_weight="balanced")),   # no probability=True
    ])


# =====================================================
# SELECT HYPERPARAMETERS (on validation only)
# =====================================================
if TUNE:
    best_score, best_params = -np.inf, None
    for params in ParameterGrid(param_grid):
        m = build(params).fit(X_train, y_train)
        s = m.decision_function(X_val)
        ap = average_precision_score(y_val, s)
        if ap > best_score:
            best_score, best_params = ap, params
    print("\nBEST PARAMS:", best_params, "| val AUPRC:", round(best_score, 4))
else:
    best_params = FIXED_PARAMS
    print("\nFIXED PARAMS:", best_params)

# =====================================================
# FINAL MODEL: refit on train+val, score test ONCE
# =====================================================
X_fit = np.vstack([X_train, X_val])
y_fit = np.concatenate([y_train, y_val])
final = build(best_params).fit(X_fit, y_fit)

scores = final.decision_function(X_test)        # one consistent score function
y_pred = (scores >= 0).astype(int)              # decision_function threshold = 0

auc   = roc_auc_score(y_test, scores)
auprc = average_precision_score(y_test, scores)
sens  = recall_score(y_test, y_pred)
prec  = precision_score(y_test, y_pred, zero_division=0)
f1    = f1_score(y_test, y_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("\n===== RESULT =====")
print(f"ROC-AUC     : {auc:.3f}")
print(f"AUPRC       : {auprc:.3f}")
print(f"Sensitivity : {sens:.3f}")
print(f"Precision   : {prec:.3f}")
print(f"F1          : {f1:.3f}")
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Baseline AUPRC (prevalence): {y_test.mean():.4f}")

Total windows: (3254, 1530)
Class counts : [3184   70]
Train/Val/Test: 2277 651 326
Test positives: 7

BEST PARAMS: {'C': 100, 'gamma': 'scale', 'k': 100} | val AUPRC: 0.845

===== RESULT =====
ROC-AUC     : 0.996
AUPRC       : 0.916
Sensitivity : 0.714
Precision   : 1.000
F1          : 0.833
TN=319 FP=0 FN=2 TP=5
Baseline AUPRC (prevalence): 0.0215


In [37]:
pip install mrmr-selection

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   --------------------------------------- 833.4/833.4 kB 12.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/52.0 MB ? eta -:--:--
   --------- ------------------------------ 12.1/52.0 MB 62.7 MB/s eta 0:00:01
   -------------------- ------------------- 26.2/52.0 MB 61.5 MB/s eta 0:00:01
   ---------------------------- ----------- 36.7/52.0 MB 58.3 MB/s eta 0:00:01
   --------------------------------- ------ 43.5/52.0 MB 51.2 MB/s eta 0:00:01
   ---------------------------------------  51.9/52.0 MB 51.6 MB/s eta 0:00:01
   ---------------------------------------- 52.0/52.0 MB 44.1 MB/s eta 0:00:00

   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   -------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Users\ritaw\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [38]:
from mrmr import mrmr_classif
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

In [39]:
class MRMRSelector(BaseEstimator, TransformerMixin):

    def __init__(self, k=50):
        self.k = k

    def fit(self, X, y):
        X_df = pd.DataFrame(X)

        self.selected_ = mrmr_classif(
            X=X_df,
            y=pd.Series(y),
            K=self.k
        )

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df[self.selected_].values

In [40]:
import numpy as np
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, confusion_matrix
)

# =====================================================
# CONFIG — the ONLY things you change between runs
# =====================================================
SEED = 42
DATA_POOL = seizure_records   # <-- swap to `records` for the all-files condition
TUNE = True                   # <-- True = grid search; False = use FIXED_PARAMS
FIXED_PARAMS = {"k": 100, "C": 10, "gamma": "scale"}

param_grid = {
    "k": [10, 30, 50, 100],
    "C": [0.1, 1, 10,100],
    "gamma": ["scale", 0.01, 0.1],
}

# =====================================================
# BUILD DATA  (identical for every run on the same pool)
# =====================================================
X_all = np.vstack([X for X, y in DATA_POOL.values()])
y_all = np.hstack([y for X, y in DATA_POOL.values()]).astype(int)
print("Total windows:", X_all.shape)
print("Class counts :", np.bincount(y_all))

# =====================================================
# SPLIT — fixed 70/20/10, fixed seed, identical every run
# =====================================================
# 1. carve off test FIRST — 10% of everything, set aside
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.10, random_state=SEED, stratify=y_all
)

# 2. split the remaining 90% into train and val
#    val = 0.20/0.90 ≈ 0.2222 of trainval  -> 20% of the whole
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=2/9, random_state=SEED, stratify=y_trainval
)
print("Train/Val/Test:", X_train.shape[0], X_val.shape[0], X_test.shape[0])
print("Test positives:", int(y_test.sum()))


def build(params):
    return Pipeline([
        ("select", MRMRSelector(k=params["k"])),
        ("scale", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=params["C"],
            gamma=params["gamma"],
            class_weight="balanced"
        ))
    ])


# =====================================================
# SELECT HYPERPARAMETERS (on validation only)
# =====================================================
if TUNE:
    best_score, best_params = -np.inf, None
    for params in ParameterGrid(param_grid):
        m = build(params).fit(X_train, y_train)
        s = m.decision_function(X_val)
        ap = average_precision_score(y_val, s)
        if ap > best_score:
            best_score, best_params = ap, params
    print("\nBEST PARAMS:", best_params, "| val AUPRC:", round(best_score, 4))
else:
    best_params = FIXED_PARAMS
    print("\nFIXED PARAMS:", best_params)

# =====================================================
# FINAL MODEL: refit on train+val, score test ONCE
# =====================================================
X_fit = np.vstack([X_train, X_val])
y_fit = np.concatenate([y_train, y_val])
final = build(best_params).fit(X_fit, y_fit)

scores = final.decision_function(X_test)        # one consistent score function
y_pred = (scores >= 0).astype(int)              # decision_function threshold = 0

auc   = roc_auc_score(y_test, scores)
auprc = average_precision_score(y_test, scores)
sens  = recall_score(y_test, y_pred)
prec  = precision_score(y_test, y_pred, zero_division=0)
f1    = f1_score(y_test, y_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("\n===== RESULT =====")
print(f"ROC-AUC     : {auc:.3f}")
print(f"AUPRC       : {auprc:.3f}")
print(f"Sensitivity : {sens:.3f}")
print(f"Precision   : {prec:.3f}")
print(f"F1          : {f1:.3f}")
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Baseline AUPRC (prevalence): {y_test.mean():.4f}")

Total windows: (3254, 1530)
Class counts : [3184   70]
Train/Val/Test: 2277 651 326
Test positives: 7


 78%|███████▊  | 78/100 [00:50<00:14,  1.53it/s]


KeyboardInterrupt: 

In [42]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

from sklearn.base import BaseEstimator, TransformerMixin
from mrmr import mrmr_classif


# =====================================================
# mRMR FEATURE SELECTOR
# =====================================================
class MRMRSelector(BaseEstimator, TransformerMixin):

    def __init__(self, k=50):
        self.k = k

    def fit(self, X, y):

        X_df = pd.DataFrame(X)

        self.selected_features_ = mrmr_classif(
            X=X_df,
            y=pd.Series(y),
            K=self.k
        )

        return self


    def transform(self, X):

        X_df = pd.DataFrame(X)

        return X_df[self.selected_features_].values



# =====================================================
# CONFIG
# =====================================================
SEED = 42

DATA_POOL = seizure_records     # swap to records for all files

TUNE = True

FIXED_PARAMS = {
    "k": 50,
    "C": 10,
    "gamma": "scale"
}


param_grid = {

    # number of mRMR selected features
    "k": [10, 30, 50, 100],

    # SVM parameters
    "C": [0.1, 1, 10],

    "gamma": [
        "scale",
        0.01
    ]
}



# =====================================================
# BUILD DATA
# =====================================================

X_all = np.vstack([
    X for X, y in DATA_POOL.values()
])

y_all = np.hstack([
    y for X, y in DATA_POOL.values()
]).astype(int)


print("Total windows:", X_all.shape)
print("Class counts :", np.bincount(y_all))



# =====================================================
# SPLIT
# =====================================================

# 10% untouched test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.10,
    random_state=SEED,
    stratify=y_all
)


# 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=2/9,
    random_state=SEED,
    stratify=y_trainval
)


print(
    "Train/Val/Test:",
    X_train.shape[0],
    X_val.shape[0],
    X_test.shape[0]
)

print(
    "Test positives:",
    int(y_test.sum())
)



# =====================================================
# PIPELINE
# =====================================================

def build(params):

    return Pipeline([

        (
            "select",
            MRMRSelector(
                k=params["k"]
            )
        ),


        (
            "scale",
            StandardScaler()
        ),


        (
            "svm",
            SVC(
                kernel="rbf",
                C=params["C"],
                gamma=params["gamma"],
                class_weight="balanced"
            )
        )

    ])




# =====================================================
# HYPERPARAMETER SEARCH
# =====================================================

if TUNE:

    best_score = -np.inf
    best_params = None


    for params in ParameterGrid(param_grid):

        print("\nTesting:", params)


        model = build(params)

        model.fit(
            X_train,
            y_train
        )


        val_scores = model.decision_function(
            X_val
        )


        ap = average_precision_score(
            y_val,
            val_scores
        )


        print(
            "Validation AUPRC:",
            round(ap,4)
        )


        if ap > best_score:

            best_score = ap
            best_params = params



    print(
        "\nBEST PARAMS:",
        best_params
    )

    print(
        "BEST VAL AUPRC:",
        round(best_score,4)
    )


else:

    best_params = FIXED_PARAMS

    print(
        "\nFIXED PARAMS:",
        best_params
    )



# =====================================================
# FINAL TRAINING
# =====================================================

X_fit = np.vstack([
    X_train,
    X_val
])

y_fit = np.concatenate([
    y_train,
    y_val
])


final = build(
    best_params
)


final.fit(
    X_fit,
    y_fit
)



# show selected features

print("\nSelected features:")
print(
    final.named_steps["select"].selected_features_
)



# =====================================================
# TEST
# =====================================================

scores = final.decision_function(
    X_test
)


# SVM boundary
y_pred = (
    scores >= 0
).astype(int)



auc = roc_auc_score(
    y_test,
    scores
)


auprc = average_precision_score(
    y_test,
    scores
)


sens = recall_score(
    y_test,
    y_pred
)


prec = precision_score(
    y_test,
    y_pred,
    zero_division=0
)


f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)


tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()



print("\n===== RESULT =====")

print(
    f"ROC-AUC     : {auc:.3f}"
)

print(
    f"AUPRC       : {auprc:.3f}"
)

print(
    f"Sensitivity : {sens:.3f}"
)

print(
    f"Precision   : {prec:.3f}"
)

print(
    f"F1          : {f1:.3f}"
)


print(
    f"TN={tn} FP={fp} FN={fn} TP={tp}"
)


print(
    f"Baseline AUPRC (prevalence): {y_test.mean():.4f}"
)

Total windows: (3254, 1530)
Class counts : [3184   70]
Train/Val/Test: 2277 651 326
Test positives: 7

Testing: {'C': 0.1, 'gamma': 'scale', 'k': 10}


100%|██████████| 10/10 [00:05<00:00,  1.68it/s]


Validation AUPRC: 0.2071

Testing: {'C': 0.1, 'gamma': 'scale', 'k': 30}


100%|██████████| 30/30 [00:20<00:00,  1.49it/s]


Validation AUPRC: 0.4608

Testing: {'C': 0.1, 'gamma': 'scale', 'k': 50}


...

Validation AUPRC: 0.6911

Testing: {'C': 10, 'gamma': 0.01, 'k': 100}


100%|██████████| 100/100 [01:01<00:00,  1.62it/s]


Validation AUPRC: 0.7882

BEST PARAMS: {'C': 10, 'gamma': 'scale', 'k': 100}
BEST VAL AUPRC: 0.7882


100%|██████████| 100/100 [00:59<00:00,  1.67it/s]



Selected features:
[915, 1195, 397, 276, 292, 286, 1022, 605, 1216, 1507, 1393, 1221, 592, 809, 1351, 280, 1280, 299, 888, 889, 598, 1511, 898, 1514, 911, 285, 277, 747, 716, 1281, 1503, 288, 609, 917, 582, 583, 1404, 1214, 727, 591, 1499, 1468, 1211, 974, 503, 904, 1005, 1478, 1205, 1347, 1522, 594, 1386, 1334, 291, 699, 1102, 1298, 1324, 1473, 1201, 897, 734, 275, 1349, 1039, 1497, 1376, 795, 671, 905, 1115, 1528, 668, 1387, 971, 910, 1523, 1508, 438, 435, 1232, 1461, 1515, 887, 630, 1052, 1230, 1456, 713, 611, 1207, 225, 1329, 877, 1263, 1328, 295, 1397, 814]

===== RESULT =====
ROC-AUC     : 1.000
AUPRC       : 1.000
Sensitivity : 0.857
Precision   : 1.000
F1          : 0.923
TN=319 FP=0 FN=1 TP=6
Baseline AUPRC (prevalence): 0.0215
